# 1.介绍

assignment1 要求手动实现：
1. BPE
2. LLM（gpt 的decoder-only）
3. cross-entropy loss & AdamW optimizer
4. train

# 2.实现

先说一下实现思路，一开始我比较懵逼，完全不知道什么意思，其实就是 adapters.py 把所有需要实现的接口都留出来了，这里以 softmax 为例：
```python
def run_softmax(in_features: Float[Tensor, " ..."], dim: int) -> Float[Tensor, " ..."]:
    """
    Given a tensor of inputs, return the output of softmaxing the given `dim`
    of the input.

    Args:
        in_features (Float[Tensor, "..."]): Input features to softmax. Shape is arbitrary.
        dim (int): Dimension of the `in_features` to apply softmax to.

    Returns:
        Float[Tensor, "..."]: Tensor of with the same shape as `in_features` with the output of
        softmax normalizing the specified `dim`.
    """
    raise NotImplementedError
```
介绍了传入的参数是什么样，你应该实现什么。

In [1]:
from __future__ import annotations
from typing import Iterable
import torch
from torch import Tensor
import torch.nn.functional as F  # 只用来对齐参考时自测，不在最终实现中依赖
import math
from typing import Iterable, Optional
from torch.optim import Optimizer
import json
import os
import regex as re
from collections.abc import Iterable, Iterator

## 2.1 工程逻辑

其实把所有的实现都写到 adapters.py 中即可，但是既然都学 Language Modeling from Scratch 了，那就按一个完整的工程来进行工作。

下面是我设计的工程目录：
```
tests/
└─ adapters.py             # 仅定义 run_*，内部调用 src/* 的实现
src/
├─ nn_utils.py             # softmax / cross_entropy / gradient_clipping
├─ optim_sched.py          # AdamW 类选择 / 余弦+warmup LR
├─ data.py                 # get_batch
├─ io.py                   # save/load checkpoint
├─ bpe/
│   ├─ tokenizer.py        # get_tokenizer
│   └─ train_bpe.py        # run_train_bpe
└─ model/
   ├─ attention.py        # sdpa / mha / rope / mha_with_rope
   ├─ layers.py           # linear / embedding / swiglu / rmsnorm
   └─ transformer.py      # block / lm
```

# 3 nn_utils

这一部分就是实现 torch 中的各种工具，其中包括：
1. Softmax
2. cross_entropy
3. gradient_clipping

## 3.1 Softmax

Softmax：多分类问题的输出层，将一组任意实数转换为一个概率分布


数学定义：假设我们有一个包含K个实数的向量 **z** = ($z_1$, $z_2$, ..., $z_K$)，Softmax函数会计算一个新的向量 **σ(z)** = ($\sigma_1$, $\sigma_2$, ..., $\sigma_K$)，其中每个元素 $\sigma_i$ 的计算公式如下：

$$\sigma(z)_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}} \quad \text{for } i = 1, \dots, K$$

在实际 PyTorch 实现中 Softmax 实现有多种优化。这里我们只说数值稳定性优化：“减去最大值技巧 (Subtract-Max Trick)”，这是Softmax实现中**最重要**的优化，用于防止计算过程中的数值上溢（overflow）和下溢（underflow）。

Softmax的核心计算是 $e^{z\_i}$。如果输入的logits向量 `z` 中包含较大的数值（例如，`z_i = 1000`），$e^{1000}$ 的结果会是一个巨大的数字，超出浮点数能表示的范围，导致**上溢 (Overflow)**，结果变为 `inf`。这会导致最终的概率分布变成 `[nan, nan, ...]`。

反之，如果logits都为非常小的负数（例如，`z_i = -1000`），$e^{-1000}$ 的结果会无限接近于0，导致**下溢 (Underflow)**。如果分子和分母都下溢为0，最终结果也会是 `nan`。

PyTorch 给输入向量的所有元素加上或减去同一个常数，其输出结果保持不变。

$$\sigma(\mathbf{z})_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}} = \frac{C \cdot e^{z_i}}{C \cdot \sum_{j=1}^{K} e^{z_j}} = \frac{e^{z_i + \log(C)}}{\sum_{j=1}^{K} e^{z_j + \log(C)}}$$

我们可以选择一个特定的常数 `C` 来优化计算。最佳选择是令 $\\log(C) = - \\max(\\mathbf{z})$，即从所有logits中减去它们的最大值。

$$\sigma(\mathbf{z})_i = \frac{e^{z_i - \max(\mathbf{z})}}{\sum_{j=1}^{K} e^{z_j - \max(\mathbf{z})}}$$

**这样做的好处是：**

1.  **防止上溢**：变换后的向量中，最大的元素是 `0` ($e^0=1$)，所有其他元素都是负数。这样就保证了指数计算的最大结果是 `1`，有效避免了上溢。
2.  **减少下溢风险**：通过将整个数值范围向上“平移”，至少保证了有一个元素（即原始最大值对应的元素）的指数结果为 `1`，使得分母至少为 `1`，从而避免了分母因所有项都下溢成零而导致的除零错误。


In [2]:
def softmax(in_features: Tensor, dim: int) -> Tensor:
    """
    数值稳定 softmax：对 in_features 沿 dim 做 softmax，输出形状与输入一致。
    关键：先减去该维最大值，避免 exp 溢出。
    """
    shifted = in_features - in_features.max(dim=dim, keepdim=True).values
    exps = torch.exp(shifted)
    return exps / exps.sum(dim=dim, keepdim=True)

## 3.2 Cross_entropy

在信息论中，**熵 (Entropy)** 用来衡量一个概率分布的“不确定性”或“信息量”。一个系统越混乱、越不可预测，它的熵就越高。

  * **直观例子**：
      * **低熵**：一枚“作弊”的硬币，99%的概率是正面。结果非常确定，所以熵很低。
      * **高熵**：一枚均匀的硬币，正反面概率各50%。结果最不确定，所以熵最高。

而**交叉熵 (Cross-Entropy)** 则更进一步，它用来衡量**两个概率分布之间的差异**。具体来说，它衡量的是，当我们**使用一个“错误的”或“近似的”概率分布 `q` 来表示一个“真实的”概率分布 `p` 时，所需要付出的额外信息量（或编码长度）**。

如果近似分布 `q` 与真实分布 `p` 非常接近，那么交叉熵的值就很低。反之，如果 `q` 与 `p` 相差甚远，交叉熵的值就会很高。

**数学定义与公式**：

假设我们有两个离散的概率分布，$p$ 和 $q$。

  * $p$: 真实分布 (True Distribution)。一般为数据的真实标签，表示为 one-hot 编码。
  * $q$: 预测分布 (Predicted Distribution)。模型的输出，通常是经过 Softmax 或 Sigmoid 函数处理后的概率。

交叉熵 $H(p, q)$ 的计算公式如下：

$$H(p, q) = - \sum_{i=1}^{K} p(x_i) \log(q(x_i))$$

其中：

  * $K$ 是所有可能事件（类别）的总数。
  * $p(x\_i)$ 是事件 $x\_i$ 在真实分布 $p$ 中的概率。
  * $q(x\_i)$ 是事件 $x\_i$ 在预测分布 $q$ 中的概率。

**注意**：我们在进行计算的时候不用完全照搬公式来计算，因为真实分布 $p$ 是 one-hot 编码的，假设真实类别是 $c$，那么只有 $p(x_c)=1$，而所有其他的 $p(x_i)=0$ (当 $i \neq c$ 时)。这样一来，上面的求和公式就可以大大简化：

$$
\begin{aligned}
H(p, q) &= -\sum_{i=1}^{V} p(x_i) \log(q(x_i)) \\
        &= -(p(x_1)\log(q(x_1)) + \dots + p(x_c)\log(q(x_c)) + \dots + p(x_V)\log(q(x_V))) \\
        &= -(0 \cdot \log(q(x_1)) + \dots + 1 \cdot \log(q(x_c)) + \dots + 0 \cdot \log(q(x_V))) \\
        &= -\log(q(x_c))
\end{aligned}
$$

所以，交叉熵损失函数最终要计算的，就是**模型预测的正确类别所对应的概率的负对数值**。我们的目标就是让这个损失值越小越好，也就是让 $q_c$ (正确类别的概率) 越接近 1 越好。


In [3]:
def cross_entropy(inputs: Tensor, targets: Tensor) -> Tensor:
    """
    稳定的交叉熵：inputs 形状 (B, V) 为未归一化 logits；targets 形状 (B,) 为 Long 类别 id。
    等价于 F.cross_entropy(inputs, targets, reduction='mean')，但用 logsumexp 保证稳定性。
    """
    # log_softmax(x) = x - logsumexp(x)
    logsumexp = torch.logsumexp(inputs, dim=1, keepdim=True)   # (B, 1)
    log_probs = inputs - logsumexp                             # (B, V)
    gathered = log_probs.gather(1, targets.view(-1, 1)).squeeze(1)  # (B,)
    return -gathered.mean()                                    # 标准 mean reduction

## 3.3 梯度裁剪 (Gradient Clipping) 

梯度裁剪是一种简单而有效的技术，用来解决梯度爆炸问题。形象地来说就是给模型设定一个“最大步长”的安全限制。

**核心思想**：在更新模型参数之前，检查所有梯度的“总长度”（即范数 L2-Norm）。

  * **如果总长度超过了你设定的阈值**：就按比例**缩小所有梯度**，使得它们的总长度刚好等于这个阈值。重要的是，**所有梯度都被同一个系数缩小**，所以梯度的**方向保持不变**，只是步子的大小被限制了。
  * **如果总长度没有超过阈值**：那说明梯度是正常的，什么也不用做。

In [4]:
def gradient_clipping(parameters: Iterable[torch.nn.Parameter], max_l2_norm: float) -> None:
    """
    全局 L2 范数裁剪：只处理 p.grad 非空的参数。若全局范数超过阈值，则用同一个系数原地缩放每个 grad。
    与 torch.nn.utils.clip_grad.clip_grad_norm_ 的语义一致。
    """
    params = [p for p in parameters if p.grad is not None]
    if not params:
        return
    device = params[0].grad.device
    # 先算每个 grad 的 2-范数，再做“范数的范数”得到全局范数
    grads_norms = torch.stack([p.grad.detach().norm(2) for p in params]).to(device)
    total_norm = grads_norms.norm(2)
    clip_coef = max_l2_norm / (total_norm + 1e-6)  # +epsilon 防 0
    if clip_coef < 1.0:
        for p in params:
            p.grad.detach().mul_(clip_coef.to(p.grad.device))  # 原地缩放

# 4 optim_sched

这一部分是实现：
1. AdamW
2. 余弦学习率调度lr_cosine_schedule

## 4.1 AdamW
```python
from torch.optim import Optimizer
```

注解：
* `Optimizer`：PyTorch 的优化器基类，继承它可自动获得**参数组（param\_groups）**、**state 管理**等通用框架。

---


> 基础数学知识
> 
> 标准 Adam（Kingma & Ba, 2014）对梯度 $g_t$ 维护一阶、二阶动量：
>
> $$
\begin{aligned}
m_t &= \beta_1 m_{t-1} + (1-\beta_1) g_t \\
v_t &= \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \\
\hat m_t &= \frac{m_t}{1-\beta_1^t},\quad
\hat v_t = \frac{v_t}{1-\beta_2^t} \\
\theta_{t+1} &= \theta_t - \alpha \frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon}
\end{aligned}
$$
>
> 传统 “L2 正则” 往往体现在把 $\lambda\theta$ 加到梯度中（即**耦合式** weight decay）。
> **AdamW（Loshchilov & Hutter, 2017）**提出把 weight decay 从梯度里**拿出来单独对参数衰减**（**解耦**），更新式变为：
>
> $$
\theta_{t+1} = \underbrace{\theta_t - \alpha\lambda \theta_t}_{\text{decoupled decay}}
\;-\; \alpha\frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon}.
$$
>
> 这能避免与自适应梯度统计的耦合带来的副作用，经验上更稳定。

In [5]:
class AdamWCustom(Optimizer):
    def __init__(
        self,
        params: Iterable[torch.nn.Parameter],
        lr: float = 1e-3,
        betas: tuple[float, float] = (0.9, 0.999),
        eps: float = 1e-8,
        weight_decay: float = 0.01,
    ):
        if le < 0.0:
            raise ValueError(f"Invalid lr:{lr}")
        if not 0.0 <= betas[0] < 1.0:
            raise ValueError(f"Invalid beta1:{betas[0]}")
        if not 0.0 <= betas[1] < 1.0:
            raise ValueError(f"Invalid beta2:{betas[1]}")
        if eps <= 0.0:
            raise ValueError(f"Invalid eps:{eps}")
        if weight_decay < 0.0:
            raise ValueError(f"Invalid weight_decay:{weifht_decay}")

        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super.__init__(params, defaults)

### `__init__`：参数合法性与默认赋值

```python
def __init__(
    self,
    params: Iterable[torch.nn.Parameter],
    lr: float = 1e-3,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
):
    if lr < 0.0:
        raise ValueError(f"Invalid lr: {lr}")
    if not 0.0 <= betas[0] < 1.0:
        raise ValueError(f"Invalid beta1: {betas[0]}")
    if not 0.0 <= betas[1] < 1.0:
        raise ValueError(f"Invalid beta2: {betas[1]}")
    if eps <= 0.0:
        raise ValueError(f"Invalid eps: {eps}")
    if weight_decay < 0.0:
        raise ValueError(f"Invalid weight_decay: {weight_decay}")

    defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
    super().__init__(params, defaults)
```

* **类型与默认值**与 `torch.optim.AdamW` 对齐：

  * `lr` 学习率，>0；
  * `betas=(β1, β2)`：动量系数，常用 (0.9, 0.999)；
  * `eps`：数值稳定项，防止分母接近 0，默认1e-8；
  * `weight_decay`：解耦权重衰减系数 $\lambda$，默认0.01。
* **参数校验**：逐个抛 `ValueError`，保证训练时不会出现“隐性错误”。
* `defaults`：优化器基类会把它复制到每个 **param group**，从而可以支持**多组**参数、不同超参。

### `step`：执行一次参数更新

**AdamW 的一次 `step` 要做什么？**

**用当前梯度更新一阶/二阶动量 → 做偏置校正 → 用解耦权重衰减 & Adam 规则更新参数**。
真正计算时的步骤：

$$
\begin{aligned}
\text{(decoupled decay)}\quad
\theta_t &\leftarrow \theta_t - \alpha \lambda \theta_t \\
m_t &= \beta_1 m_{t-1} + (1-\beta_1) g_t \\
v_t &= \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \\
\hat m_t &= \frac{m_t}{1-\beta_1^t},\qquad
\hat v_t = \frac{v_t}{1-\beta_2^t} \\
\theta_{t+1} &= \theta_t - \alpha \frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon}
\end{aligned}
$$

其中 $g_t=\nabla_\theta \mathcal{L}(\theta_t)$，$\alpha$ 学习率，$\lambda$ 权重衰减系数，$\beta_1,\beta_2$ 动量指数衰减，$\epsilon$ 数值稳定项。

In [6]:
    @torch.no_grad()
    def step(self, closure: Optional[callable] = None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr: float = group["lr"]
            beta1, beta2 = group["betas"]
            eps: float = group["eps"]
            weight_decay: floar = group["weight_decay"]

            for p in group["params"]:
                if p.grad is None:
                    continue
                grad = p.grad

                if grad.is_sparse:
                    raise RuntimeError("AdamWCustom does not support sparse gradients")

                state = self.state[p]
                if len(state) == 0:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(p, memory_format=torch.preserve_format)
                    state["exp_avg_sq"] = torch.zero_like(p, memory_format=torch.preserve_format)

                exp_avg: torch.Tensor = state["exp_avg"]
                exp_avg_sq: torch.Tensor = state["exp_avg_sq"]

                state["step"] += 1
                step: int = state["step"]

                if weight_decay != 0.0:
                    p.add_(p, alpha=-lr * weight_decay)

                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                bias_correction1 = 1.0 - beta1 ** step
                bias_correction2 = 1.0 - beta2 ** step
                step_size = lr / bias_correction1

                denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(eps)

                p.addcdiv_(exp_avg, denom, value=-step_size)
        return loss

**1. 装饰器与 closure**

```python
@torch.no_grad()
def step(self, closure: Optional[callable] = None):
    loss = None
    if closure is not None:
        with torch.enable_grad():
            loss = closure()
```

* `@torch.no_grad()`：**更新参数不需要被 autograd 记录**，否则会把优化步骤也纳入计算图，既慢又错。
* `closure`：pytorch中的一个函数，封装了 一次完整的前向 + 反向传播，是为了兼容像 LBFGS 这类“需要在 step 内重新计算 loss”的优化器接口保留的惯例。
  AdamW中用不到，这里只是为了保持和原版 pytorch 实现一致：若传入 `closure`，短暂 `enable_grad` 来重新前反传一次，拿到 `loss` 返回。


**2. 遍历参数组与超参获取**

```python
for group in self.param_groups:
    lr: float = group["lr"]
    beta1, beta2 = group["betas"]
    eps: float = group["eps"]
    weight_decay: float = group["weight_decay"]
```

* **param\_groups** 是 `torch.optim.Optimizer` 的核心机制，允许**不同层或张量用不同超参**（不同 lr / wd 等）。
* 读出本组的学习率、动量系数、eps、权重衰减。



**3. 遍历参数与梯度校验**

```python
for p in group["params"]:
    if p.grad is None:
        continue
    grad = p.grad

    if grad.is_sparse:
        raise RuntimeError("AdamWCustom does not support sparse gradients")
```

* `None` 冻结参数或本轮未参与计算的部分直接跳过。
* AdamW（官方实现也是）**不支持稀疏梯度**（稀疏需要特殊更新公式）。


**4. 状态（state）初始化 & 时间步**

```python
state = self.state[p]
if len(state) == 0:
    state["step"] = 0
    state["exp_avg"] = torch.zeros_like(p, memory_format=torch.preserve_format)
    state["exp_avg_sq"] = torch.zeros_like(p, memory_format=torch.preserve_format)

exp_avg: torch.Tensor = state["exp_avg"]
exp_avg_sq: torch.Tensor = state["exp_avg_sq"]

state["step"] += 1
step: int = state["step"]
```

* state 是一个**字典**，专门用来存储某个参数 p 在优化过程中的**历史状态信息**(动量、平方梯度、步数等)。
* 首次遇到参数：

  * `step=0`（马上会自增为 1）
  * `exp_avg`（$m_t$）初始 0 张量
  * `exp_avg_sq`（$v_t$）初始 0 张量
* `memory_format=torch.preserve_format`：**保留原参数内存布局**（例如 `channels_last`）。利于后续算子 kernel 最优化与缓存局部性。
* `step` 记录时间步 $t$，供**偏置校正**使用（$\beta^t$）。


**5. 解耦权重衰减（Decoupled Weight Decay）**

```python
if weight_decay != 0.0:
    p.add_(p, alpha=-lr * weight_decay)
```

* 这是 AdamW 和“把 L2 正则直接加到梯度里”的**关键区别**：

  * **解耦**：直接对参数做缩放 $\theta \leftarrow \theta - \alpha \lambda \theta = (1-\alpha\lambda)\theta$；
  * **耦合（L2 正则）**：把 $\lambda \theta$ 加到梯度上，交给后面的自适应分母去缩放，这会改变衰减在各维度上的相对强度（与 $v_t$ 相关），**经验上更不稳定**。
* 为什么放在动量更新**之前**？
  PyTorch AdamW 也是这么做的；从数值角度讲，放前或放后**几乎等价**（差在浮点细节）。

> 工程经验：很多训练(比如LayerNorm 和 bias 参数）会**关闭 LN/偏置项的衰减**（把它们放进一个 `weight_decay=0` 的组里），用 param\_groups 很容易做到。


**6. 指数滑动平均（EMA）的一阶/二阶动量**

```python
exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
```

* 公式对应：

  $$
  m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t,\qquad
  v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2.
  $$
* `addcmul_` 是 fused 原地：`exp_avg_sq += (1-β2) * (grad * grad)`；
  `mul_` / `add_` 都是**原地**，减少中间张量，省内存、快。
* **直觉**：

  * $m_t$：平滑的“平均梯度方向”
  * $v_t$：平滑的“各维度梯度方差”，用来**自适应**缩放学习率（维度越“剧烈”，步子越小）。


**7. 偏置校正（Bias Correction）**

```python
bias_correction1 = 1.0 - beta1 ** step
bias_correction2 = 1.0 - beta2 ** step
step_size = lr / bias_correction1
```

* 因为 $m_0=v_0=0$，前期 $m_t,v_t$ 有向 0 的偏置。校正项：

  $$
  \hat m_t = \frac{m_t}{1-\beta_1^t},\quad
  \hat v_t = \frac{v_t}{1-\beta_2^t}.
  $$
* 把 $1-\beta_1^t$ 放到 `step_size` 里（$\alpha/(1-\beta_1^t)$），把 $1-\beta_2^t$ 放到分母的 `sqrt` 里——**与官方实现完全等价**、更数值稳定。


**8. 分母（自适应因子）与最终更新**

```python
denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(eps)
p.addcdiv_(exp_avg, denom, value=-step_size)
```

* `denom = sqrt(v_hat) + eps`，其中 $v\_hat = v_t/(1-\beta_2^t)$。
  代码写成 $\sqrt{v_t}/\sqrt{1-\beta_2^t}$ 是一样的（更稳定的因式分解）。
* `add_(eps)`：$+\epsilon$ 防止分母接近 0（比如早期或梯度很小的情况）。
* `addcdiv_`：原地做 `p -= step_size * exp_avg / denom`。
  注意我们用的是**未除偏的一阶动量 `exp_avg`**，配合 `step_size = lr/(1-β1^t)`，这正是 $\alpha \hat m_t$。


**9. 返回 loss（如果有 closure）**

```python
return loss
```

* 若上面调用过 `closure`，把它的标量 `loss` 传回去，方便上层记录。

In [7]:
def get_adamw_cls():
    return AdamWCustom

返回“我们自己实现的AdamW 类”，语义上是对其 torch.optim.AdamW 的。

## 4.2 lr_cosine_schedule

虽然这里函数名只有 cosine_schedule ，并且 pytorch 官方的余弦学习率中也没有，但是实现要求里说明了需要带 linear warmup。
```python
"""
Given the parameters of a cosine learning rate decay schedule (with linear
warmup) and an iteration number, return the learning rate at the given
iteration under the specified schedule.
"""
```

**1. 什么是 warmup?**

**warmup** 指 **学习率预热**（learning rate warmup）：在训练的最初几个epoch，学习率不会直接设到目标的最大值，而是**从一个较小的值开始，逐渐增大到预设的最大学习率**。

常见形式：

* **线性 warmup**：学习率从 0 线性增加到最大值。
* **指数 warmup**：学习率按指数规律增加。
* **常见实践**：先 warmup 几百或几千个 step，再进入正常的学习率调度（例如余弦衰减、step decay 等）。



**2. 为什么要用 warmup？**

训练初期模型参数是随机初始化的，网络输出和梯度分布都很不稳定，如果一开始就用一个**很大的学习率**，容易造成：

* **梯度爆炸 / 发散**：初始梯度方向杂乱，大学习率会导致参数更新过大，训练不稳定；
* **损失震荡甚至 nan**：尤其是 Transformer / 大模型，层归一化 + 残差结构容易放大这种不稳定性；
* **优化器的动量未收敛**：像 Adam/AdamW 等自适应优化器一开始的动量估计还不准，大步更新会导致方向错误。

因此 warmup 的核心目的：

* **稳定训练初期的优化过程**
* **让优化器的动量有时间收敛**
* **避免大学习率直接冲击随机初始化参数**



**3. 举个例子**

假设我们设定最大学习率 = 0.001

* 如果没有 warmup，第一步就直接 lr=0.001，梯度可能把参数“甩飞”，训练不稳定。
* 如果用 1000 step 的线性 warmup：

  * step=1 → lr=0.000001
  * step=500 → lr=0.0005
  * step=1000 → lr=0.001
  * 之后才开始余弦衰减。

这样，前 1000 step 相当于“热身”，让模型逐渐适应大学习率。

In [8]:
def get_lr_cosine_schedule(
    *,
    it: int,
    max_learning_rate: float,
    min_learning_rate: float,
    warmup_iters: int,
    cosine_cycle_iters: int,
) -> float:
    if it <= warmup_iters:
        if warmup_iters <= 0:
            return float(max_learning_rate)
        return float(max_learning_rate) * (it / float(warmup_iters))

    span = max(1, cosine_cycle_iters - warmup_iters)
    progress = (it - warmup_iters) / float(span)
    if progress >= 1.0:
        return float(min_learning_rate)

    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    lr = min_learning_rate + (max_learning_rate - min_learning_rate) * cosine

    if lr < min_learning_rate:
        lr = min_learning_rate
    if lr > max_learning_rate:
        lr = max_learning_rate

    return float(lr)

**1. 线性 warmup（含端点）**

```python
    # 线性 warmup（含端点）
    if it <= warmup_iters:
        if warmup_iters <= 0:
            return float(max_learning_rate)
        return float(max_learning_rate) * (it / float(warmup_iters))
```

* 当 `it` 还没超过 warmup 终点：

  * 特判 `warmup_iters <= 0`：这表示**根本不做 warmup**。这里直接返回 `max_learning_rate`，等价于“warmup 长度为 0 时从第一步就用 `max_lr`”（避免除零）。
  * 正常情况：线性插值

    $$
    \text{lr}(it) = \alpha_{\max} \cdot \frac{it}{T_w},\quad 0 \le it \le T_w.
    $$

    * `it=0` ⇒ `0`；`it=warmup_iters` ⇒ `max_learning_rate`。
    * “包含端点”的处理让 **warmup 结束那一刻**恰好到达峰值 LR，和社区常见实现一致。


**2. 余弦退火**

```python
    # 余弦段
    span = max(1, cosine_cycle_iters - warmup_iters)  # 防止除零
    progress = (it - warmup_iters) / float(span)
    if progress >= 1.0:
        return float(min_learning_rate)
```

* 进入余弦段的前提是 `it > warmup_iters`（上一段已 return 结束），此时需要把 `it` 归一化到 $(0,1)$。
* `span = max(1, T_c - T_w)`：

  * 余弦段长度是 `T_c - T_w`。若写错参数导致 `T_c <= T_w`，这里用 `max(1, …)` 防止除零——**稳健性**考虑。
* `progress = (it - T_w) / span`：

  * 当 `it = T_w + 1` 时 `progress` 接近 `0^+`；
  * 当 `it` 逐步逼近 `T_c`，`progress → 1^-`。
* `if progress >= 1.0:`：

  * 当 `it >= T_c`（或某些极端舍入）直接落到**尾段**：返回 `min_learning_rate`。

```python
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    lr = min_learning_rate + (max_learning_rate - min_learning_rate) * cosine
```

* 经典的**半周期余弦**系数（从 1 平滑到 0）：

  $$
  \text{cosine}(t) = \tfrac{1}{2}\left(1 + \cos(\pi t)\right),\quad t\in[0,1].
  $$

  * `t=0` ⇒ `cosine=1`，学习率等于 `max_lr`；
  * `t=1` ⇒ `cosine=0`，学习率等于 `min_lr`。
* 线性映射到区间 $[\alpha_{\min}, \alpha_{\max}]$：

  $$
  \text{lr}(it) = \alpha_{\min} + (\alpha_{\max}-\alpha_{\min}) \cdot \text{cosine}(\text{progress}).
  $$
* 这保证了**连续性**：在 warmup 末端到余弦起点，LR 值连续（不过导数在分段点一般**不连续**，这是常见且可接受的设计）。


**3. 数值钳制与返回**

```python
    # 数值钳制，避免浮点误差越界
    if lr < min_learning_rate:
        lr = min_learning_rate
    if lr > max_learning_rate:
        lr = max_learning_rate
    return float(lr)
```

* 由于浮点误差，`cos`/除法可能让结果在端点附近出现极细小的**越界**（例如应为 0.1 却算成 0.09999999997）。
* 这里对 LR 做**上下界钳制**，保证严格落在 $[\alpha_{\min}, \alpha_{\max}]$。
* `float(lr)`：确保返回的是 Python `float`（不是 numpy 标量或别的类型）。这在某些日志/序列化路径上更稳。

# 5 data

这一部分是实现：    
get_batch

就是实现：从一个长的一维 dataset 流中，随机抽取 batch_size 个连续窗口，每个窗口长度为 context_length；X 是窗口本身，Y 是右移一位（next-token 预测的标签）     
关于 pin_memory：
1. 关闭 pin_memory：在 GPU 显存足够的情况下，可以关闭 pin_memory，该方法将 dataset 一次性全部加载到 GPU 上所有的计算都是在 GPU 上完成，CPU-GPU 的数据传输仅进行一次，所以性能最好；
2. 开启 pin_memory：在 GPU 显存不足以容纳整个数据集的情况下，将数据集保留在 CPU 主内存中，每次只在 CPU 上准备好一个小批量的数据，然后仅将这个小批量数据传输到 GPU。该方法引入了每一步的 CPU-GPU 数据传输开销，但大大降低了对显存的需求，保证了程序的可用性。此外，通过使用 pin_memory（锁页内存）的优化方法，可以使数据传输异步进行，从而与 GPU 上的计算并行，能最大限度地隐藏数据传输带来的延迟。

In [9]:
def get_batch(
    *, dataset_np, batch_size: int, context_length: int,
    device: str, pin_memory: bool = False
):
    is_cuda = str(device).startswith("cuda")
    if pin_memory and not is_cuda:
        raise ValueError("pin_memory=True only makes sense when device is CUDA")

    if pin_memory:
        # CPU + pinned 路径
        toks = torch.as_tensor(dataset_np, dtype=torch.long, device="cpu").pin_memory()
        idx_device = "cpu"
    else:
        # 直接放目标设备
        toks = torch.as_tensor(dataset_np, dtype=torch.long, device=device)
        idx_device = device  # 索引必须与 toks 在同设备

    n = toks.numel()
    if n < context_length + 1:
        raise ValueError(f"need at least {context_length+1}, got {n}")

    max_start = n - context_length - 1
    starts = torch.randint(0, max_start + 1, (batch_size,), device=idx_device)
    ar = torch.arange(context_length, device=idx_device)
    idx = starts[:, None] + ar[None, :]
    X = toks[idx]
    Y = toks[idx + 1]

    if pin_memory:
        # 仅在 pinned 路径下做异步 H2D
        X = X.to(device, non_blocking=True)
        Y = Y.to(device, non_blocking=True)

    return X, Y

# 6 I/O Checkpoints

这一部分是实现：    
1. save_checkpoint
2. load_checkpoint

## 6.1 _is_pathlike

区分“路径”和“文件对象”

In [10]:
def _is_pathlike(x) -> bool:
    return isinstance(x, (str, bytes, os.PathLike))

## 6.2 save_checkpoint

**把训练的中间状态保存到 checkpoint 文件**，方便后续恢复。

保存的内容包括：

1. **模型参数 (`model.state_dict()`)**
   只包含模型里权重和 buffer（例如 `LayerNorm.running_mean`），而不是整个模型对象。这样更稳健，跨版本/跨路径都能加载。

2. **优化器状态 (`optimizer.state_dict()`)**
   包含动量项、平方梯度累积等（Adam/AdamW 中的 `exp_avg`、`exp_avg_sq`），否则恢复训练时梯度动态会断裂。

3. **当前迭代数 (`iteration`)**
   用来恢复训练循环的步数、学习率调度器位置、日志编号等。

              
> 这里说明一下为什么直接用 torch.save()，作业说明 cs336_spring2025_assignment1_basics.pdf 中明确说了对于模型原理相关的 torch.nn, torch.nn.functional, 或 torch.optim 中的大部分定义不能使用，但是5.2部分说了torch.save(obj, dest) can dump an object ... which can then be loaded back into memory with torch.load(src). 所以是没问题的。

In [11]:
def save_checkpiont(
    *, 
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    iteration: int,
    out: str | os.PathLike | BinaryIO | IO[bytes],
) -> None:
    payload = {
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "iteration": int(iteration),
    }
    if _is_pathlike(out):
        with open(out, "wb") as f:
            torch.save(payload, f)
    else:
        torch.save(payload, out)

## 6.3 load_checkpoint

1. **读取 checkpoint 文件**

   * 支持文件路径（字符串、`Path`）
   * 也支持已经打开的二进制文件对象（例如 `open(..., "rb")` 或 `io.BytesIO()`）

   通过 `torch.load(..., map_location="cpu")` 把保存的内容读进来（先放到 CPU，保证跨设备加载安全）。

2. **恢复训练状态**

   * 用 `model.load_state_dict(payload["model_state"])` 把模型的参数恢复到保存时的值。
   * 用 `optimizer.load_state_dict(payload["optimizer_state"])` 把优化器的动量、学习率组等内部状态也恢复回来。
     这样模型和优化器都会回到**和保存时一模一样的状态**。

3. **返回迭代计数**

   * 从 checkpoint 中取出 `iteration` 并返回。
   * 训练循环里就能接着跑，比如从第 1000 步恢复继续往下训练。

In [13]:
def load_checkpoint(
    *,
    src: str | os.PathLike | BinaryIO | IO[bytes],
    model: torch.nn.Model,
    optimizer: torch.optim.Optimizer,
) -> int:
    if _is_pathlike(src):
        with open(src, "rb") as f:
            payload = torch.load(f, map_location = "cpu")
    else:
        payload = torch.load(src, map_location = "cpu")

    model.load_state_dict(payload["model_state"])
    optimizer.load_state_dict(payload["optimizer_state"])

    return int(payload["interation"])

# 7 Tokenizer



这一部分实现的是：

**BPE（Byte Pair Encoding）**：**字节对编码**算法。我们要创建一个“分词器”（Tokenizer）。语言模型（比如GPT）不直接理解文字，它们只理解数字。分词器的任务就是把人类的语言（比如字符串 "Hello, world!"）转换成一串数字（比如 [15496, 11, 995, 0]），并且能再把这串数字转换回原来的文字。     

BPE 的**核心思想**：一开始把所有单个的字符当作基础词汇，然后不断地寻找最常出现的相邻“词对”，把它们合并成一个新的“词”。重复这个过程，就能学到像 "ing", "tion" 这样常见的词根或单词片段。这样既能有效压缩词汇表大小，又能处理没见过的词（OOV, Out-of-Vocabulary a problem）。

如果想要深入了解 BPE，非常推荐去看这个教程：[Let's build GPT: from scratch, in code, spelled out.](https://www.youtube.com/watch?v=kCc8FmEb1nY&list=PLwHkaSK8WYmI67jO5EEYtzTyVOT9i5GWB&index=2)

In [2]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

```python
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
```

这是 **GPT-2 分词器的预分词（pre-tokenization）正则表达式**。它的作用是：把原始文本切成“初步的词块”（pieces），然后再交给 BPE 规则去进一步合并。

1. **`'(?:[sdmt]|ll|ve|re)`**

   * 匹配英语里的常见缩写后缀，例如：

     * `I'm` → `'m`
     * `you're` → `'re`
     * `they've` → `'ve`
     * `she'll` → `'ll`
     * `it's` → `'s`
   * 用 `?:` 表示非捕获分组。

2. **` ?\p{L}+`**

   * `\p{L}` 表示 Unicode 中的“字母”（Letter）。
   * `+` 表示连续的一个或多个字母。
   * 前面的 ` ?` 允许前面有一个空格。
   * 意义：**字母串（单词），并保留开头空格**。
   * 例子：

     * `"Hello"` → `"Hello"`
     * `" world"` → `" world"`

3. **` ?\p{N}+`**

   * `\p{N}` 表示 Unicode 中的“数字”（Number）。
   * 类似上面的逻辑，表示一串数字，可以前面带一个空格。
   * 例子：

     * `"123"` → `"123"`
     * `" 42"` → `" 42"`

4. **` ?[^\s\p{L}\p{N}]+`**

   * 匹配“非空格、非字母、非数字”的连续字符（符号/标点），前面允许有一个空格。
   * 例子：

     * `","` → `","`
     * `" !!!"` → `" !!!"`

5. **`\s+(?!\S)`**

   * `\s+` 表示一个或多个空白字符。
   * `(?!\S)` 表示**后面不是非空白**，即这些空格后面没有别的东西了（字符串结尾）。
   * 意义：匹配**尾部的空格**。

6. **`\s+`**

   * 匹配一般的空格（如果前面几条规则都没处理）。

**总结**

这个正则的作用是把文本切分成几类 token 初始块：

1. **缩写后缀**：`'s`, `'ll`, `'ve`, `'re`, `'m`, `'t`, `'d`
2. **单词**：前导空格 + 字母串
3. **数字**：前导空格 + 数字串
4. **符号/标点**：前导空格 + 一串标点/符号
5. **尾部空格**
6. **其他空格**

**举个例子**

输入文本：

```
Hello, I'm 25 years old!  
```

经过这个正则分词：

```
["Hello", ",", " I", "'m", " 25", " years", " old", "!", "  "]
```

之后这些块会被转换成字节序列，再交给 BPE 合并。

In [3]:
class Tokenizer:
    def __init__(
        self,
        vocab: dict[int, bytes],
        merges: list[tuple[bytes, bytes]],
        spcial_tokens: list[str] | None = None,
    ):
        self.vocab = vocab
        self.encoder = {b: i for i, b in vocab.items()}
        self.bpe_ranks = {pair: i for i, pair in enumerate(merges)}
        self.special_tokens = special_tokens or []
        self.special_tokens_encoder: dict[str, int] = {}
        self.special_tokens_decoder: dict[int, str] = {}
        self.pat = re.compile(PAT)
        self.special_pat = None
        if self.special_tokens:
            self._setup_special_tokens()

```python
# 1. 存储基础数据
self.vocab = vocab
self.encoder = {b: i for i, b in vocab.items()}
```
1.  **`self.vocab` 和 `self.encoder`**:

      * `vocab`: “解码器”，一个从数字ID到字节（bytes）的字典。当我们有了一个ID，比如 `50256`，可以用 `vocab[50256]` 查到它代表的字节。
      * `encoder`: “编码器”，它正好和`vocab`相反。我们用它来把字节（比如 `b'<|endoftext|>'`）转换成对应的ID。这里用了一个字典推导式 `{b: i for i, b in vocab.items()}` 来快速创建这个反向映射。这样，编码和解码的查询都非常快。

```python
self.bpe_ranks = {pair: i for i, pair in enumerate(merges)}
```
2.  **`self.bpe_ranks`**:

      * `merges` 是一个列表，包含了所有合并规则，并且是按学习顺序排列的。例如 `[(b'e', b'n'), (b'en', b'd')]`。
      * 因为在编码时需要频繁地查找需要优先合并的词对。在列表中搜索效率很低,可以通过将其转换成一个字典 `bpe_ranks`，键是词对 `(b'e', b'n')`，值是它在列表中的索引 rank。**rank 越小，优先级越高**。这样可以将查找一个词对的优先级的时间复杂度变成 O(1) 。

```python
self.special_tokens = special_tokens or []
self.special_tokens_encoder: dict[str, int] = {}
self.special_tokens_decoder: dict[int, str] = {}
``` 
3.  **`self.special_tokens` 和 `self.special_tokens_encoder/decoder`**:

      * 有一些特殊 token 我们不希望被BPE算法拆分，比如 `"<|endoftext|>"` (文本结束符)。
      * 将这些特殊 token 单独存起来，并创建一个专用的编码器和解码器。

```python
self.pat = re.compile(PAT)
self.special_pat = None
```
4.  **`self.pat = re.compile(PAT)`**:

      * `PAT` 前面已经说了，这里不再赘述。
      * `re.compile()` 会预编译这个正则表达式。（如果我们要在一个循环里反复使用同一个正则表达式，预编译可以大大提高速度）

```python
if self.special_tokens:
    self._setup_special_tokens()
```
5.  **`self._setup_special_tokens()`**:

      * 专门用来处理特殊 token,我们稍后会详细看它。

In [4]:
    def _setup_special_tokens(self) -> None:
        sorted_special_tokens = sorted(self.special_tokens, key=len, reverse=True)
        for token_str in sorted_special_token:
            token_bytes = token_str.encode("utf-8")
            if token_bytes not in self.encoder:
                new_id = len(self.vocab)
                self.vocab[new_id] = token_bytes
                self.encoder[token_bytes] = new_id
            self.special_tokens_encoder[token_str] = self.encoder[token_bytes]
            self.special_tokens_decoder[self.encoder[token_bytes]] = token_str
        special_pattern = "|".join(re.escape(st) for st in sorted_special_tokens)
        self.special_pat = re.compile(f"({special_pattern})")

```python
sorted_special_tokens = sorted(self.special_tokens, key=len, reverse=True)
```
1. **`sorted_special_tokens`**:

   * 将所有的特殊 token 按照长度进行排序，保证子集重叠的 token 能够先处理较长一条，比如：

      * `"<|eot|>"`
      * `"<|eot|><|eot|>"`

   * 那么必须先匹配 **更长的** `" <|eot|><|eot|>"`，否则会被前者提前切开。
   * 所以正则里优先级由长度保证。

```python
# 将特殊token添加到词汇表中
for token_str in sorted_special_tokens:
    token_bytes = token_str.encode("utf-8")
    if token_bytes not in self.encoder:
        # 分配一个新的ID
        new_id = len(self.vocab)
        self.vocab[new_id] = token_bytes
        self.encoder[token_bytes] = new_id
    
    # 存储特殊token的字符串到ID的映射
    self.special_tokens_encoder[token_str] = self.encoder[token_bytes]
    self.special_tokens_decoder[self.encoder[token_bytes]] = token_str
```
2. **把特殊 token 加入词表，并存储其映射**

   * 首先将新的 token 编码
   * 然后为其分配一个新的 id（因为词表 `self.vocab` 原本是 **id → bytes**，`self.encoder` 是 **bytes → id**），新 id = 当前 vocab 的长度
   * 最后再将特殊字符串 **id → str** 和 **str → id** 的映射保存到 special_tokens_encoder 和 special_tokens_decoder

> 这里解释一下，特殊字符之所以特殊是因为没办法直接用 utf-8 进行编解码，所以需要存储一个专门的编码解码器用来对其进行编码和解码。

```python
# 创建一个正则表达式，用于根据特殊token分割输入文本
# 使用re.escape来处理可能包含正则表达式元字符的特殊token
special_pattern = "|".join(re.escape(st) for st in sorted_special_tokens)
self.special_pat = re.compile(f"({special_pattern})")
```
3. **构造正则，用来切分文本**

   * 构建一个正则表达式，这个表达式能一次性在文本中找到所有我们定义的特殊 token 。
   * 当使用一个正则表达式去调用 re.split() 时，它不仅会按匹配到的内容分割字符串，还会把匹配到的内容（也就是我们的特殊token）本身也保留在结果列表里